# Format single `known_genotypes.xsxl` file into individual DNA profiles

In [1]:
import pandas as pd
from DNAnet.allele_callers import NON_AUTOSOMAL_MARKERS

In [2]:
# Load the Excel file
df = pd.read_excel('resources/data/ProvedIt/PROVEDIt_RD14-0003_GF_Known_Genotypes.xlsx')

In [3]:
df['Sample ID']
for sample_name, group in df.head(1).groupby('Sample ID'):
    print(f"Sample ID: {sample_name}")
    print(f"group: {group}")
    print()

Sample ID: 1
group:   Research ID  Sample ID D3S1358    vWA D16S539 CSF1PO  TPOX  Yindel AMEL  \
0   RD14-0003          1   14,18  17,19   12,13  11,12  8,11     NaN  X,X   

  D8S1179  ...    FGA D22S1045  D5S818 D13S317 D7S820     SE33 D10S1248  \
0   12,15  ...  20,25    11,19   11,12    8,12    9,9  21,31.2    14,14   

  D1S1656 D12S391 D2S1338  
0   14,15   21,23   20,22  

[1 rows x 26 columns]



In [4]:
# # For each sample (row) in the DataFrame:


# for idx, row in df.iterrows():
#     sample_name = row['Sample ID']
#     # Exclude the Sample ID and Research ID columns to get marker columns
#     marker_columns = [col for col in df.columns if not col in ['Sample ID', 'Research ID'] and not col in NON_AUTOSOMAL_MARKERS]
#     data = []
#     for marker in marker_columns:
#         alleles = str(row[marker]).split(',')
#         # Handle missing or malformed data
#         allele1 = alleles[0] if len(alleles) > 0 else ''
#         allele2 = alleles[1] if len(alleles) > 1 else ''
#         data.append([sample_name, marker, allele1, allele2])
#     # Create the new DataFrame for this sample
#     new_df = pd.DataFrame(data, columns=['Sample ID', 'Marker', 'Allele1', 'Allele2'])
#     print(new_df)  # Or save to CSV: 
#     new_df.to_csv(f"resources/data/ProvedIt/References/{sample_name}.csv", sep=';', index=False)

## Test the python file code

In [5]:
from DNAnet.data.kit_compatibility.format_conversion import individualize_genotypes

!pwd

# Use the individualize_genotypes function to process the file
individualize_genotypes(
    input_path='resources/data/ProvedIt/PROVEDIt_RD14-0003_GF_Known_Genotypes.xlsx',
    file_type='excel',
    exclude_columns=['Sample ID', 'Research ID'] + NON_AUTOSOMAL_MARKERS,
    output_dir='resources/data/ProvedIt/References',
)

/Users/amarmesic/Documents/tudelft/thesis/DNANet


# Redo logic to obtain contributors to each .hid file
I have added this to [utils.py](DNAnet/utils.py)

In [6]:
import re

In [10]:
# Example filename to extract contributors and proportions
filename = "A02_RD14-0003-31_32-1;1-M2c-0.03GF-Q2.0_01.15sec.hid"
filename = "this_is_not a valid filename"

match = re.search(r"RD14-0003-(\d+)_(\d+)-(\d+);(\d+)", filename)
if match:
    c1, c2 = match.group(1), match.group(2)
    r1, r2 = match.group(3), match.group(4)
    print(f"Contributors: {c1}, {c2}")
    print(f"Proportions: {r1}, {r2}")

print(match)

None


In [11]:
from collections import defaultdict
import csv
from DNAnet.data.data_models.dna_models import Allele, Marker, Panel
from DNAnet.utils import get_contributors_from_filename


def load_donor_alleles_provedit(file_name: str, panel: Panel) -> list[Marker]:
    """
    For R&D files, we know the donors that contributed and the DNA profiles of the donors. For a
    single .hid file, find the donors (from the file name) and return the list of Markers of those
    donors combined.
    :param file_name: .hid file to load actual donors for
    :param rd_data_root: root folder of the RD data, containing a Referenties folder
    :param panel: the panel to retrieve the dye row of the markers from
    """
    reference_path = "resources/data/ProvedIt/References"
    
    contributors = get_contributors_from_filename(file_name)

    # find the set of all alleles of the donors per marker
    marker_allele_strings = defaultdict(set)
    for file_stem in contributors:
        reference_profiles_path = os.path.join(reference_path, f'{file_stem}.csv')
        with open(reference_profiles_path, "r") as f:
            reader = csv.DictReader(f, delimiter=";")
            for row in reader:
                marker_allele_strings[row['Marker']].update([row['Allele1'], row['Allele2']])

    # transform into Marker/Allele objects
    markers = []
    for marker_name, alleles in marker_allele_strings.items():
        dye_row = panel.get_dye_row(marker_name)
        markers.append(Marker(dye_row, marker_name, [Allele(a) for a in sorted(alleles)]))
    return markers

In [ ]:
load_donor_alleles_provedit(